<a href="https://colab.research.google.com/github/MSahitya/LandCostPrediction-SupervisedML/blob/main/PlotPricePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install polars lightgbm xgboost joblib

In [13]:
import polars as pl
import numpy as np
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# Simulate 20 years of historical data (e.g., 2006 to 2026)
num_records = 25000
start_date = datetime(2006, 1, 1)

dates = [start_date + timedelta(days=int(np.random.randint(0, 365*20))) for _ in range(num_records)]
regions = np.random.choice(['North', 'South', 'East', 'West', 'Central'], size=num_records)
land_size = np.random.uniform(500, 10000, size=num_records) # in sq ft
distance_to_center = np.random.uniform(1, 50, size=num_records) # in km

# Generate target (land cost) influenced by size, location, inflation trend, and noise
base_price = 50
inflation_factor = np.array([(d - start_date).days / 365.25 for d in dates]) * 7.5 # yearly appreciation
cost = (land_size * (base_price + inflation_factor) * (50 / (distance_to_center + 5))) + np.random.normal(0, 5000, num_records)
cost = np.clip(cost, 10000, None) # Min price floor

# Build Polars DataFrame
df = pl.DataFrame({
    "date": dates,
    "region": regions,
    "land_size_sqft": land_size,
    "distance_to_center_km": distance_to_center,
    "land_cost": cost
})

# Simulate some missing data using a safe numpy mask (3% missing data)
missing_mask = np.random.random(num_records) < 0.03

df = df.with_columns([
    pl.when(pl.lit(missing_mask))
    .then(None)
    .otherwise(pl.col("distance_to_center_km"))
    .alias("distance_to_center_km")
])

print(df.head())

shape: (5, 5)
┌─────────────────────┬─────────┬────────────────┬───────────────────────┬───────────────┐
│ date                ┆ region  ┆ land_size_sqft ┆ distance_to_center_km ┆ land_cost     │
│ ---                 ┆ ---     ┆ ---            ┆ ---                   ┆ ---           │
│ datetime[μs]        ┆ str     ┆ f64            ┆ f64                   ┆ f64           │
╞═════════════════════╪═════════╪════════════════╪═══════════════════════╪═══════════════╡
│ 2025-11-27 00:00:00 ┆ Central ┆ 5671.146558    ┆ 27.9504               ┆ 1.7253e6      │
│ 2008-05-10 00:00:00 ┆ Central ┆ 8802.423219    ┆ 25.63604              ┆ 974166.657571 │
│ 2020-10-04 00:00:00 ┆ Central ┆ 6922.02789     ┆ 38.9566               ┆ 1.2635e6      │
│ 2020-04-23 00:00:00 ┆ East    ┆ 4507.711248    ┆ 30.06267              ┆ 1.0113e6      │
│ 2020-03-19 00:00:00 ┆ East    ┆ 6676.314914    ┆ 13.194178             ┆ 2.8770e6      │
└─────────────────────┴─────────┴────────────────┴───────────────────────┴──

In [14]:
# Safer production approach: only drop if it exists
if "date" in df.columns:
    df = df.with_columns([
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.month().alias("month"),
    ]).drop("date")

# Separate features (X) and target (y)
X = df.drop("land_cost").to_pandas() # Convert to standard pandas for Scikit-Learn pipeline compatibility
y = df["land_cost"].to_numpy()

# Split into chronological Train/Test split (e.g., train on first 18 years, test on final 2 years)
split_year = 2024
train_idx = X["year"] < split_year
test_idx = X["year"] >= split_year

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb

# Define processing strategies for different columns
numeric_features = ['land_size_sqft', 'distance_to_center_km', 'year', 'month']
categorical_features = ['region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Natively addresses missing data
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Encodes categorical regions safely
])

# Combine into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Bundle preprocessing and the XGBoost Regressor engine together
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42))
])

# Train the model prototype
model_pipeline.fit(X_train, y_train)
print("Prototype pipeline successfully trained!")

Prototype pipeline successfully trained!


In [16]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Generate predictions
y_pred = model_pipeline.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"--- Prototype Evaluation ---")
print(f"RMSE (Root Mean Squared Error): ${rmse:,.2f}")
print(f"MAE (Mean Absolute Error): ${mae:,.2f}")
print(f"R² Score: {r2:.4f}")

--- Prototype Evaluation ---
RMSE (Root Mean Squared Error): $375,851.51
MAE (Mean Absolute Error): $188,961.42
R² Score: 0.9690


In [17]:
import joblib
from google.colab import files

# Save the unified pipeline file locally within the environment
model_filename = "land_cost_predictor_pipeline.joblib"
joblib.dump(model_pipeline, model_filename)
print(f"Model successfully saved as {model_filename}")

# Optional: Download the model artifact directly to your local computer
files.download(model_filename)

Model successfully saved as land_cost_predictor_pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>